# Feature Engineering — Leuven ↔ Brussels Commuter Delays (Checkpoint)

**Business goal:** warn Leuven↔Brussels commuters that their train is going to be delayed, roughly one hour ahead of time.

**Where we left off.** Data Understanding gave us a clean, per-stop-event table for the six corridor stations, Jan–Jul 2026, with weather already joined and a `touches_leuven` flag. Two findings from that stage matter a lot here:

- **Delay propagation is strong** — a train that's late at one corridor stop is, almost one-for-one, still late at the next one. That's the single most useful signal a 1-hour-ahead model can lean on.
- **The corridor is small and fast** — most runs cross all six stations in well under an hour, and a lot of runs only clip 1-2 of them. That shapes how much "upstream" history we'll actually have for a given prediction.

This is CRISP-DM step 3: **Feature Engineering**, and it's the step where it's easiest to accidentally cheat. A model that's allowed to peek at information from *after* the moment it's supposed to make a prediction will look great in a notebook and fail completely in production; the real system simply doesn't have that information yet when it has to make a call. We'll be strict about that throughout.

In [1]:
import os

# make "../../data/..." paths resolve whether Jupyter was launched from the repo root or from notebooks/
if not os.path.isdir("../../data") and os.path.isdir("data"):
    os.chdir("notebooks/assignment_notebooks")

In [2]:
import os
import pandas as pd
import numpy as np

CLEAN_CORRIDOR_PATH = "../../data/checkpoints/clean_leuven_corridor.parquet"
TRAIN_PATH = "../../data/checkpoints/train.parquet"
VAL_PATH = "../../data/checkpoints/val.parquet"
TEST_PATH = "../../data/checkpoints/test.parquet"

RUN_KEYS = ["TRAIN_NO", "service_date", "RELATION"]  # identifies one physical train run

corridor = pd.read_parquet(CLEAN_CORRIDOR_PATH)
corridor = corridor.loc[~corridor["delay_arr_is_outlier"]].copy()
corridor.shape

(890100, 33)

## 🤔 At a prediction time *t*, what is the target?

**Target:** `delay_arr_min` at Leuven, which states the arrival delay in minutes, framed as **regression** (predict "how many minutes late", not just "late or not"). We only define a target for train runs where `touches_leuven` is True and Leuven has a real arrival recorded (i.e. Leuven isn't that run's very first stop, which by construction has no arrival to predict).

**Time *t*:** the business wants a warning "roughly one hour ahead". In practice, the earliest moment we could actually issue a warning for a given run is right after we observe its most recent corridor stop *before* Leuven, that's when fresh, run-specific information starts existing. We'll come back to how close that is to a full hour once we've measured it below.

**Why regression, not classification?** A binary framing, such as "will this train be more than 5 minutes late?", is a perfectly reasonable alternative, and arguably closer to what a commuter-facing warning needs (a yes/no push notification, not a number to the minute). We pick regression here because it's strictly more informative: a predicted number of minutes can always be thresholded into a yes/no warning after the fact (at whatever threshold the product team wants), but a classifier's yes/no can't be turned back into minutes. It also gives Stage 4 more room to build a genuinely useful baseline.

## 🤔 What data is actually available for prediction *at time t*?

For a given train run, the only rows we're allowed to look at are the ones for corridor stops **strictly before** Leuven in that run's stop order, plus schedule information for Leuven itself (the timetable is fixed and known well in advance, it's not an outcome). Concretely:

- **Allowed:** any corridor stop this run passed before reaching Leuven (its delay, its weather at the time, which station it was); the *scheduled* arrival time at Leuven (for calendar features); the train type/service line (fixed for the whole run, known at departure).
- **Not allowed:** anything actually observed at Leuven itself (its real arrival/departure time, its weather, its delay) other than the target we're predicting; anything from a corridor stop that comes *after* Leuven in the run.

First we need each run's stop order. We sort by the **scheduled** time (`planned_arrival`, falling back to `planned_departure` for a run's first stop) rather than the realized time. The physical order of stations on the line doesn't change with how delayed a train is, and the schedule is known in advance, so ordering by it can't leak anything either.

In [3]:
corridor["sched_time"] = corridor["planned_arrival"].fillna(corridor["planned_departure"])

corridor = corridor.sort_values(RUN_KEYS + ["sched_time"], kind="mergesort").reset_index(drop=True)
corridor["stop_idx"] = corridor.groupby(RUN_KEYS).cumcount()  # 0-based position of this stop within its run

corridor[RUN_KEYS + ["station", "sched_time", "stop_idx"]].head(8)  # stop_idx resets to 0 at the start of each new run

,TRAIN_NO,service_date,RELATION,station,sched_time,stop_idx
0,10,2026-01-01,ICE,Leuven,2026-01-01 21:11:00,0
1,10,2026-01-01,ICE,Kortenberg,2026-01-01 21:17:00,1
2,10,2026-01-01,ICE,Brussels-North,2026-01-01 21:26:00,2
3,10,2026-01-01,ICE,Brussels-Central,2026-01-01 21:31:00,3
4,10,2026-01-01,ICE,Brussels-South,2026-01-01 21:35:00,4
5,10,2026-01-02,ICE,Leuven,2026-01-02 21:11:00,0
6,10,2026-01-02,ICE,Kortenberg,2026-01-02 21:17:00,1
7,10,2026-01-02,ICE,Brussels-North,2026-01-02 21:26:00,2


In [4]:
is_target = (
    (corridor["station"] == "Leuven")
    & (corridor["touches_leuven"])
    & (corridor["real_arrival"].notna())
)
print(f"{is_target.sum()} rows touch Leuven with a recorded real arrival")

no_upstream = is_target & (corridor["stop_idx"] == 0)
print(f"{no_upstream.sum()} of those have stop_idx == 0 that means Leuven is the FIRST corridor station this run touches "
      f"(e.g. a train arriving from outside our 6-station scope, so it has a real arrival without ever passing "
      f"through an earlier corridor stop). We can't build any upstream feature for these, so they can't be used here.")

is_target = is_target & (corridor["stop_idx"] > 0)
print(f"{is_target.sum()} usable target rows remain")

86838 rows touch Leuven with a recorded real arrival
53409 of those have stop_idx == 0 that means Leuven is the FIRST corridor station this run touches (e.g. a train arriving from outside our 6-station scope, so it has a real arrival without ever passing through an earlier corridor stop). We can't build any upstream feature for these, so they can't be used here.
33429 usable target rows remain


## 🤔 How do we transform available data into features?

For every row we can compute, from the *previous* row in the same run (i.e. strictly upstream of it), a handful of signals: the delay observed there, a running summary of delay so far, which station it was, and the weather at the time. Because we compute these for every row via `shift`/`expanding` *within each run*, they're automatically leakage-safe for any row we later select as a target, the value at row *i* only ever depends on rows before *i* in that same run.

On top of the upstream signal we add:
- **calendar features** from Leuven's *scheduled* arrival time (hour of day, day of week, month) known well in advance, not an outcome;
- **weather** at the last upstream stop (not at Leuven itself as Leuven's own weather is only known once the train is basically there, which is information from the future relative to time *t*);
- **train type**, extracted from `RELATION` (e.g. `IC 03` → `IC`), constant for the whole run and known at departure.

In [5]:
corridor["stop_delay_min"] = corridor["delay_arr_min"].fillna(corridor["delay_dep_min"])
by_run = corridor.groupby(RUN_KEYS)

corridor["upstream_stop_count"] = corridor["stop_idx"]
corridor["last_stop_station"] = by_run["station"].shift(1)
corridor["last_stop_delay_min"] = by_run["stop_delay_min"].shift(1)
corridor["mean_upstream_delay_min"] = by_run["stop_delay_min"].transform(lambda s: s.shift(1).expanding().mean())
corridor["max_upstream_delay_min"] = by_run["stop_delay_min"].transform(lambda s: s.shift(1).expanding().max())
corridor["last_stop_sched_time"] = by_run["sched_time"].shift(1)

for col in ["temperature_2m", "precipitation", "rain", "wind_speed_10m", "wind_gusts_10m", "weather_bucket"]:
    corridor[f"last_stop_{col}"] = by_run[col].shift(1)

corridor.loc[is_target, ["station", "last_stop_station", "last_stop_delay_min", "mean_upstream_delay_min", "upstream_stop_count"]].head(8)

,station,last_stop_station,last_stop_delay_min,mean_upstream_delay_min,upstream_stop_count
1065,Leuven,Kortenberg,-0.750000,-0.208333,4
1070,Leuven,Kortenberg,0.400000,0.512500,4
1075,Leuven,Kortenberg,0.383333,0.216667,4
1080,Leuven,Kortenberg,0.600000,0.554167,4
1085,Leuven,Kortenberg,0.316667,0.775000,4
1090,Leuven,Kortenberg,2.433333,2.345833,4
1095,Leuven,Kortenberg,1.166667,0.470833,4
1100,Leuven,Kortenberg,1.400000,1.329167,4


In [6]:
corridor["sched_hour"] = corridor["sched_time"].dt.hour
corridor["sched_dow"] = corridor["sched_time"].dt.day_name()
corridor["sched_month"] = corridor["sched_time"].dt.month
corridor["train_type"] = corridor["RELATION"].str.extract(r"^([A-Za-z]+)")[0]
corridor["scheduled_minutes_since_last_stop"] = (corridor["sched_time"] - corridor["last_stop_sched_time"]).dt.total_seconds() / 60

corridor.loc[is_target, "scheduled_minutes_since_last_stop"].describe()

count    33429.000000
mean         9.021647
std          3.969993
min          3.100000
25%          6.000000
50%          8.000000
75%         10.000000
max         42.000000
Name: scheduled_minutes_since_last_stop, dtype: float64

The last upstream corridor stop sits only ~9 minutes before Leuven's scheduled arrival on average (median 8, and only a handful of runs anywhere close to a full hour). That's a real limitation to flag honestly: with data scoped to these 6 stations, "the freshest information available before Leuven" is not actually a full hour ahead for most runs. A production system aiming for a genuine 1-hour lead time would need visibility further upstream than our corridor covers. We proceed anyway since it's the best leakage-safe signal available in this dataset, and note the caveat for Stage 5.

In [7]:
FEATURE_COLS = [
    "upstream_stop_count", "last_stop_station", "last_stop_delay_min",
    "mean_upstream_delay_min", "max_upstream_delay_min", "scheduled_minutes_since_last_stop",
    "last_stop_temperature_2m", "last_stop_precipitation", "last_stop_rain",
    "last_stop_wind_speed_10m", "last_stop_wind_gusts_10m", "last_stop_weather_bucket",
    "sched_hour", "sched_dow", "sched_month", "train_type",
]
ID_COLS = ["TRAIN_NO", "service_date", "RELATION"]
TARGET_COL = "delay_arr_min"

features = corridor.loc[is_target, ID_COLS + ["source_month"] + FEATURE_COLS + [TARGET_COL]].copy()
print(features.shape)
features.isna().sum()

(33429, 21)


TRAIN_NO                             0
service_date                         0
RELATION                             0
source_month                         0
upstream_stop_count                  0
last_stop_station                    0
last_stop_delay_min                  0
mean_upstream_delay_min              0
max_upstream_delay_min               0
scheduled_minutes_since_last_stop    0
last_stop_temperature_2m             4
last_stop_precipitation              4
last_stop_rain                       4
last_stop_wind_speed_10m             4
last_stop_wind_gusts_10m             4
last_stop_weather_bucket             4
sched_hour                           0
sched_dow                            0
sched_month                          0
train_type                           0
delay_arr_min                        0
dtype: int64

A handful of rows are missing the upstream weather columns: the same late-night, last-day-of-collection boundary effect Data Understanding already flagged for the weather join. Too few to matter; we drop them rather than impute anything.

In [8]:
features = features.dropna(subset=["last_stop_temperature_2m"])
features.shape

(33425, 21)

## 🤔 How do we split train/validation/test?

**We reject a random row-level split** (e.g. `train_test_split(features, test_size=0.2, random_state=0)`). This is time-series data: each row is a train run anchored to a specific moment in time, and nearby runs are correlated with each other (same week, same weather system, same timetable). A random split scatters runs from every month across train/val/test alike, so validation and test rows routinely sit right next to training rows in time which is letting the model implicitly "see the future" relative to any given validation point, which inflates validation/test performance in a way that will not hold up once the model is actually deployed and only ever sees genuinely future data.

Instead we split **chronologically** using `source_month`, matching how the model will really be used: trained on the past, evaluated on data that comes strictly after it:

- **train:** January–April
- **validation:** May
- **test:** June–July

We drop `source_month` from the saved tables afterwards: it exactly encodes which split a row belongs to (Jan-Apr rows are only ever in train, etc.), so keeping it around would just hand the model a free, meaningless-in-production way to tell splits apart. We already extracted the genuinely useful part of it into the `sched_month` feature.

In [9]:
TRAIN_MONTHS = ["202601", "202602", "202603", "202604"]
VAL_MONTHS = ["202605"]
TEST_MONTHS = ["202606", "202607"]

train = features.loc[features["source_month"].isin(TRAIN_MONTHS)].drop(columns="source_month").reset_index(drop=True)
val = features.loc[features["source_month"].isin(VAL_MONTHS)].drop(columns="source_month").reset_index(drop=True)
test = features.loc[features["source_month"].isin(TEST_MONTHS)].drop(columns="source_month").reset_index(drop=True)

assert len(train) + len(val) + len(test) == len(features)
print(f"train: {len(train):>6} rows  ({train['TRAIN_NO'].min()!r}..)")
print(f"val:   {len(val):>6} rows")
print(f"test:  {len(test):>6} rows")
train.head()

train:  18666 rows  ('11'..)
val:     4665 rows
test:   10094 rows


,TRAIN_NO,service_date,RELATION,upstream_stop_count,last_stop_station,last_stop_delay_min,mean_upstream_delay_min,max_upstream_delay_min,scheduled_minutes_since_last_stop,last_stop_temperature_2m,last_stop_precipitation,last_stop_rain,last_stop_wind_speed_10m,last_stop_wind_gusts_10m,last_stop_weather_bucket,sched_hour,sched_dow,sched_month,train_type,delay_arr_min
0,11,2026-01-01,ICE,4,Kortenberg,-0.750000,-0.208333,0.033333,6.0,1.1,0.0,0.0,20.5,43.2,clear,6,Thursday,1,ICE,-1.933333
1,11,2026-01-02,ICE,4,Kortenberg,0.400000,0.512500,0.766667,6.0,1.4,0.3,0.1,16.8,39.6,snow,6,Friday,1,ICE,-0.516667
2,11,2026-01-03,ICE,4,Kortenberg,0.383333,0.216667,0.416667,6.0,-1.1,0.0,0.0,15.9,37.8,cloudy,6,Saturday,1,ICE,-0.783333
3,11,2026-01-04,ICE,4,Kortenberg,0.600000,0.554167,1.233333,6.0,-1.7,0.0,0.0,15.9,32.8,clear,6,Sunday,1,ICE,-0.533333
4,11,2026-01-05,ICE,4,Kortenberg,0.316667,0.775000,1.600000,6.0,-3.5,0.0,0.0,13.4,26.6,mostly clear,6,Monday,1,ICE,-0.783333


## Save the Feature Tables

Same feature columns in all three tables, plus the target (`delay_arr_min`) and the identifying columns (`TRAIN_NO`, `service_date`, `RELATION`) Stage 4 will need for error analysis.

In [10]:
os.makedirs("../data/checkpoints", exist_ok=True)
train.to_parquet(TRAIN_PATH, index=False)
val.to_parquet(VAL_PATH, index=False)
test.to_parquet(TEST_PATH, index=False)

for name, path in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)]:
    check = pd.read_parquet(path)
    print(name, check.shape)
check.dtypes

train (18666, 20)
val (4665, 20)
test (10094, 20)


TRAIN_NO                                        str
service_date                         datetime64[us]
RELATION                                        str
upstream_stop_count                           int64
last_stop_station                               str
last_stop_delay_min                         float64
mean_upstream_delay_min                     float64
max_upstream_delay_min                      float64
scheduled_minutes_since_last_stop           float64
last_stop_temperature_2m                    float64
last_stop_precipitation                     float64
last_stop_rain                              float64
last_stop_wind_speed_10m                    float64
last_stop_wind_gusts_10m                    float64
last_stop_weather_bucket                        str
sched_hour                                    int32
sched_dow                                       str
sched_month                                   int32
train_type                                      str
delay_arr_mi